# Chapter 22: Reserve Variability

Code of Chapter 22 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

The walkthrough comes first, because it loads the data and imports used by the other examples of the chapter.

## 22.7 Python Walkthrough

In [ ]:
import numpy as np
import pandas as pd
from gitriangle import triangle, synthetic_history
from givar import mack, odp_bootstrap, summarize

# Section 22.2.3: the example triangle
T = pd.DataFrame([[1000, 1500, 1650, 1700], [1100, 1700, 1850, np.nan],
                  [1200, 1750, np.nan, np.nan], [1300, np.nan, np.nan, np.nan]],
                 index=[2022, 2023, 2024, 2025], columns=[12, 24, 36, 48])
M = mack(T)
M["se"].round(2), round(M["total_se"], 2)                    # 4.68, 18.44, 73.53; 78.08

# Section 22.2.4 and 22.3: synthetic paid triangle
_, claims, tx = synthetic_history()
pt = triangle(claims, tx, "paid", 2021, 2025)
mack(pt)["total_se"]                                         # 258122
boot = odp_bootstrap(pt, n_sims=10_000, seed=22)
summarize(boot["reserves"]).round(0)                         # mean 2173623 ... q99.5 2877097

## Exercise 1: Process or parameter?

In [ ]:
import numpy as np
import pandas as pd
from givar import mack
T = pd.DataFrame([[1000, 1500, 1650, 1700], [1100, 1700, 1850, np.nan],
                  [1200, 1750, np.nan, np.nan], [1300, np.nan, np.nan, np.nan]],
                 index=[2022, 2023, 2024, 2025], columns=[12, 24, 36, 48])
M = mack(T)
f, s2, full = M["f"].to_numpy(), M["sigma2"].to_numpy(), M["full"].to_numpy()
den = [T.iloc[:3 - j, j].sum() for j in range(3)]
U = full[3, -1]
proc = U**2 * sum(s2[j] / f[j]**2 / full[3, j] for j in range(3))
par = U**2 * sum(s2[j] / f[j]**2 / den[j] for j in range(3))
np.sqrt(proc), np.sqrt(par)                                  # 61.93, 39.64 (thousands)

## Exercise 3: Fewer simulations

In [ ]:
from gitriangle import synthetic_history, triangle
from givar import odp_bootstrap
_, claims, tx = synthetic_history()
pt = triangle(claims, tx, "paid", 2021, 2025)
[round(np.quantile(odp_bootstrap(pt, 1_000, seed=s)["reserves"], 0.995)) for s in range(1, 6)]